# BAN với BEATs: mixture → FRCRN masking → BEATs

Phân loại 36 lớp nhiễu trên mixture giọng nói VLSP và nhiễu AudioSet, dùng **Born-Again Networks (BAN)** với BEATs làm backbone.

Khác với tài liệu đặc tả (mục 1 coi FRCRN là ngoài phạm vi), phiên bản này **đưa mixture qua FRCRN masking trước khi vào BEATs**, theo yêu cầu của dự án:

- FRCRN học mask phổ để ước lượng giọng nói `ŝ`. Phần còn lại `n̂ = x − ŝ` là nhiễu, đây là đầu vào phân loại.
- Teacher và student **dùng chung một front-end FRCRN đóng băng**, nên cùng nhìn một đầu vào. Đây là điều kiện để soft target của teacher có ý nghĩa cho student.
- Teacher là BEATs đã fine-tune trên `best.pt`. Student là BEATs khởi tạo từ pretrain, huấn luyện với loss BAN.

Sơ đồ tổng quan:

![Pipeline BAN với BEATs](figures/ban-with-beat/pipeline_overview.svg)

| Bước | Nội dung |
| --- | --- |
| 1 | Tên pipeline, chế độ chạy, thiết bị |
| 2 | Cấu hình `imp/ban-with-beat/config/ban_beat.yaml` |
| 3 | Dataset `mixture-36l`, kiểm tra không rò rỉ giữa các tập |
| 4 | Teacher: FRCRN + BEATs, đóng băng |
| 5 | Student: front-end chung + BEATs |
| 6 | Một batch đi qua pipeline: shape, loss, gradient |
| 7 | Huấn luyện các thế hệ, đánh giá, BANE |
| 8 | Bảng kết quả và biểu đồ theo SNR |

`RUN_TRAINING = False` (mặc định) chạy smoke test nhỏ vào `output/test/`. Đặt `True` để chạy thật vào `output/`.

In [ ]:
# Chạy từ thư mục gốc của repo để import được `scripts`, `src` và gói `ban_beat` nằm trong imp/ban-with-beat
import os
import sys
from pathlib import Path

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "rules.MD").is_file())
PACKAGE_ROOT = ROOT / "imp" / "ban-with-beat"
os.chdir(ROOT)
for path in (ROOT, PACKAGE_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
ROOT

## (1) Tên pipeline, chế độ chạy, thiết bị

`RUN_TRAINING = False` chạy smoke test: số epoch, số mẫu và số thế hệ đều được thu nhỏ (xem `smoke_config`). Kết quả ghi vào `output/test/`.

In [ ]:
# (1) Tên pipeline, chế độ chạy, thiết bị và thư mục đầu ra
import torch

from ban_beat.config import PACKAGE_ROOT, REPO_ROOT, load_config, smoke_config
from ban_beat.pipeline import run_pipeline

PIPELINE_NAME = "ban-beat"
RUN_TRAINING = False  # True: chạy thật vào output/; False: smoke test nhỏ vào output/test/
RUN_ID = None  # ví dụ "ban-beat-2026-10-29-10-00-00" để chạy tiếp một lần dở dang

CONFIG_PATH = PACKAGE_ROOT / "config" / "ban_beat.yaml"
OUTPUT_DIR = REPO_ROOT / "output" if RUN_TRAINING else REPO_ROOT / "output" / "test"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"{PIPELINE_NAME} | RUN_TRAINING={RUN_TRAINING} | device={device} | output={OUTPUT_DIR}")

## (2) Cấu hình

File `config/ban_beat.yaml` gồm các mục sau:

| Mục | Ý nghĩa | Giá trị hiện tại |
| --- | --- | --- |
| `teacher.finetune` | `true`: FRCRN `frcrn_checkpoint` đóng băng, BEATs khởi tạo từ pretrain gốc rồi fine-tune từ đầu trên train; `false`: nạp `teacher.checkpoint` | `true` |
| `teacher.frcrn_checkpoint` | Checkpoint FRCRN dùng làm front-end | `pretrained/FRCRN/best_e037_20261020T203958Z.pt` |
| `teacher.input_rms` | Mức RMS đưa vào BEATs sau FRCRN, `null` là lấy từ checkpoint | `null` |
| `loss.kd_weight` (λ) | Tỉ lệ soft target trong loss | 0.8 (giá trị thí nghiệm, không phải của bài BAN) |
| `loss.temperature` (T) | Nhiệt độ làm mềm phân phối | 2.0 |
| `loss.attention_weight` (β) | Attention transfer trên hidden state các layer `attention_layers`; 0 là tắt, bài gốc dùng 1000 | 0.0 (tắt) |
| `train` | Optimizer, LR, scheduler, số epoch, early stop, `amp` | AdamW, lr 5e-5, cosine, bf16 |
| `generations` | Số thế hệ student, seed, BANE | 2 thế hệ, seed 42, 43, 44 |
| `baseline` | Ensemble không chưng cất, cùng số model | tắt |

Các giá trị huấn luyện được đặt trong ô code bên dưới (`TRAIN_OVERRIDES`, `LOSS_OVERRIDES`, `GENERATION_OVERRIDES`, `BASELINE_OVERRIDES`) và ghi đè lên yaml. Muốn đổi cấu hình, sửa ở ô này; file yaml chỉ là mặc định cho CLI. `smoke_config` chỉ thu nhỏ epoch, số mẫu và số thế hệ, không đổi công thức loss hay front-end.

In [ ]:
# (2) Cấu hình huấn luyện đặt tại đây; các mục không có ở đây lấy từ config/ban_beat.yaml
from dataclasses import replace

from ban_beat.config import load_config, smoke_config

TRAIN_OVERRIDES = {
    "lr": 5e-5,
    "weight_decay": 0.01,
    "epochs": 30,
    "patience": 6,
    "batch_size": 128,
    "eval_batch_size": 64,
    "amp": "bf16",
    "seed": 42,
    "selection_metric": "macro_f1_snr",
    "label_smoothing": 0.1,
    "plateau_factor": 0.5,
    "plateau_patience": 2,
}
LOSS_OVERRIDES = {"kd_weight": 0.8, "temperature": 2.0, "attention_weight": 0.0, "attention_layers": (4, 8, 12)}
GENERATION_OVERRIDES = {"count": 2, "seeds": (42, 43, 44), "student_init": "scratch", "ensemble": True}
BASELINE_OVERRIDES = {"enabled": False}
DATA_OVERRIDES = {"nhat_cache_dir": None}

base = load_config(CONFIG_PATH)
config = replace(
    base,
    train=replace(base.train, **TRAIN_OVERRIDES),
    loss=replace(base.loss, **LOSS_OVERRIDES),
    generations=replace(base.generations, **GENERATION_OVERRIDES),
    baseline=replace(base.baseline, **BASELINE_OVERRIDES),
    data=replace(base.data, **DATA_OVERRIDES),
)
if not RUN_TRAINING:
    config = smoke_config(config)
print("model_config:", config.model_config.name, "| teacher:", config.teacher.checkpoint)
print("train:", config.train.to_dict())
print("loss:", config.loss.to_dict())
print("generations:", config.generations.to_dict())

## (3) Dataset `mixture-36l` và kiểm tra rò rỉ

Mỗi clip là hỗn hợp giọng nói và nhiễu ở một mức SNR trong `{-5, 0, 5, 10, 15, 20}` dB, cắt hoặc đệm về 4 giây. Mỗi mẫu có `sample_id`, `noise_ytid` (clip nhiễu gốc) và `speech_id`.

- `check_split_disjoint` dừng pipeline nếu train và validation có chung `sample_id`, `noise_ytid` hoặc `speech_id`. Việc tách theo clip nhiễu gốc quan trọng vì một clip được trộn ở nhiều mức SNR.
- `coverage_report` kiểm tra số mẫu theo lớp và theo SNR, và liệt kê lớp không có mẫu.
- Tập `test` không được nạp trong notebook này.

In [ ]:
# (3) Nạp mixture-36l, kiểm tra tách tập, thống kê lớp và SNR
from ban_beat.data import build_datasets, check_split_disjoint, coverage_report

datasets = build_datasets(config.data)
overlaps = check_split_disjoint(datasets)
coverage = coverage_report(datasets, config.data.snr_levels)
print({split: len(dataset) for split, dataset in datasets.items()})
print("chồng lấn (phải bằng 0):", overlaps)
print("SNR train:", coverage["snr_counts"]["train"])
print("lớp thiếu:", coverage["missing_classes"])

## (4) Teacher

Hai nhánh:

- **`teacher.finetune: true`** (mặc định trong `config/ban_beat.yaml`): không nạp checkpoint teacher. Front-end là FRCRN `frcrn_checkpoint` (đóng băng). BEATs được khởi tạo từ pretrain gốc `BEATs_iter3_plus_AS2M.pt` và fine-tune từ đầu trên tập train ngay trong ô (7). Ô này chỉ dựng front-end.
- **`teacher.finetune: false`**: nạp checkpoint đã có (`teacher.checkpoint`, ví dụ `best.pt` gồm FRCRN + BEATs) bằng `load_teacher`.

Đường đi của front-end:

1. `FRCRNSeparator(x)` trả về `n̂ = x − ŝ`, phần nhiễu sau khi khử giọng.
2. `rms_normalize(n̂, input_rms)` đưa về cùng mức RMS mà BEATs đã thấy khi huấn luyện.
3. `BEATClassifier` cho logits 36 lớp.

Sơ đồ FRCRN masking:

![FRCRN masking](figures/ban-with-beat/frcrn_masking.svg)

In [ ]:
# (4) Front-end (và teacher nếu không fine-tune): nhánh finetune chỉ dựng FRCRN, còn lại huấn luyện ở ô (7)
from ban_beat.config import resolve_path
from ban_beat.teacher import DEFAULT_INPUT_RMS, load_teacher
from src.model.frcrn import FRCRNSeparator

if config.teacher.finetune:
    teacher = None
    separator = None
    if config.data.nhat_cache_dir is None:
        separator = FRCRNSeparator.from_yaml(resolve_path(config.teacher.frcrn_config))
        separator.load_pretrained(resolve_path(config.teacher.frcrn_checkpoint))
        separator = separator.to(device).freeze()
    input_rms = config.teacher.input_rms if config.teacher.input_rms is not None else DEFAULT_INPUT_RMS
    print("finetune | FRCRN:", resolve_path(config.teacher.frcrn_checkpoint).name, "| input_rms:", input_rms)
else:
    teacher, teacher_metadata = load_teacher(
        resolve_path(config.teacher.checkpoint),
        config.model_config,
        resolve_path(config.teacher.frcrn_config),
        device,
        config.teacher.input_rms,
    )
    separator, input_rms = teacher.separator, teacher.input_rms
    print(teacher_metadata.to_dict())
print("front-end:", type(separator).__name__)

## (5) Student: front-end chung và BEATs

`BANStudent` ghép hai phần:

- **Front-end**: đúng là `separator` và `input_rms` của teacher, đóng băng và giữ ở chế độ eval, kể cả khi student ở chế độ train (BatchNorm trong UNet không cập nhật).
- **BEATs**: khởi tạo từ `BEATs_iter3_plus_AS2M.pt` (self-supervised AudioSet), gắn đầu 36 lớp mới, huấn luyện toàn bộ.

Lý do dùng chung front-end: BAN giả định teacher và student nhìn cùng đầu vào. Nếu student nhận waveform thô còn teacher nhận nhiễu đã khử giọng, soft target sẽ không mô tả đúng đầu vào của student.

Sơ đồ encoder BEATs:

![Encoder BEATs](figures/ban-with-beat/beats_encoder.svg)

In [ ]:
# (5) Dựng student: front-end chung (FRCRN đóng băng) và BEATs khởi tạo từ pretrain
from ban_beat.pipeline import build_student

student = build_student(config, seed=int(config.generations.seeds[0]), separator=separator, input_rms=input_rms).to(device)
trainable = sum(p.numel() for p in student.parameters() if p.requires_grad) / 1e6
frozen = sum(p.numel() for p in student.parameters() if not p.requires_grad) / 1e6
print(f"student: {trainable:.2f}M tham số huấn luyện | {frozen:.2f}M đóng băng (front-end FRCRN)")
print("front-end dùng chung:", student.separator is separator)
print(f"BEATs: {student.config.encoder_layers} khối, {student.config.num_classes} lớp")

## (6) Một batch đi qua pipeline

Kiểm tra theo thứ tự: shape của mixture, residual nhiễu, RMS sau chuẩn hoá, logits của teacher và student, loss BAN, và gradient. Gradient phải đi vào BEATs của student và không đi vào FRCRN.

Hàm mất mát:

$$
L = (1 - \lambda)\,\mathrm{CE}(y, p_s) + \lambda\,T^2\,\mathrm{KL}(p_t \,\|\, p_s) + \frac{\beta}{2 |\mathcal{A}|}\sum_{l \in \mathcal{A}} \left\| \hat{a}^{s}_l - \hat{a}^{t}_l \right\|_2^2,
\qquad p_t = \mathrm{softmax}(z_t / T),\quad p_s = \mathrm{softmax}(z_s / T)
$$

- `y` là nhãn cứng, `z_t` và `z_s` là logits của teacher và student.
- Nhân `T²` để gradient của thành phần soft không nhỏ đi khi tăng `T`.
- `λ = 0` là huấn luyện thông thường không có teacher.
- Số hạng cuối là attention transfer (AT). `a_l` là bản đồ năng lượng theo token của hidden state layer `l` (trung bình bình phương theo chiều đặc trưng, chuẩn hoá L2 theo token), `𝒜` là `attention_layers`; số hạng AT là trung bình theo layer, không cộng tổng. Teacher không nhận gradient. AT chỉ bật khi `β > 0`; khi đó teacher chạy trực tiếp trên mỗi batch thay vì dùng cache logits.

Sơ đồ một bước huấn luyện:

![Một bước huấn luyện](figures/ban-with-beat/ban_training_step.svg)

In [ ]:
# (6) Một batch qua front-end; nếu có teacher thì kiểm tra thêm logits, loss BAN và gradient
from ban_beat.data import build_loaders
from ban_beat.loss import BornAgainLoss
from ban_beat.runtime import rms_normalize

loaders = build_loaders(
    datasets,
    batch_size=4,
    eval_batch_size=4,
    num_workers=0,
    balanced_sampling=False,
    seed=config.train.seed,
)
batch = next(iter(loaders["validation"]))
waveform = batch["waveform"].to(device)
labels = batch["target"].argmax(dim=1).to(device)
print("mixture:", tuple(waveform.shape), "| SNR:", batch["snr_db"].tolist())

with torch.no_grad():
    residual = waveform if separator is None else separator(waveform)
    normalized = rms_normalize(residual, input_rms) if input_rms is not None else residual
print("n̂ (nhiễu sau FRCRN):", tuple(residual.shape), "| RMS trước chuẩn hoá:", round(residual.square().mean().sqrt().item(), 4))
print("RMS sau chuẩn hoá:", round(normalized.square().mean(dim=-1).sqrt().mean().item(), 4))

if teacher is None:
    print("teacher chưa có (finetune): logits teacher và loss BAN được kiểm tra trong ô (7)")
else:
    with torch.no_grad():
        teacher_logits = teacher(waveform)
    print("logits teacher:", tuple(teacher_logits.shape))
    criterion = BornAgainLoss(
        kd_weight=config.loss.kd_weight,
        temperature=config.loss.temperature,
        label_smoothing=config.train.label_smoothing,
        confidence_weighting=config.loss.confidence_weighting,
        confidence_power=config.loss.confidence_power,
    )
    student.train()
    logits = student(waveform).logits
    loss = criterion(logits, teacher_logits, labels)
    loss.backward()
    print("loss:", round(loss.item(), 4), "| hữu hạn:", bool(torch.isfinite(loss)))
    print("gradient vào BEATs của student:", student.classifier.head.fc.weight.grad is not None)
    print("gradient vào FRCRN:", any(p.grad is not None for p in student.separator.parameters()))
    student.zero_grad(set_to_none=True)

## (7) Huấn luyện các thế hệ, đánh giá và BANE

`run_pipeline` thực hiện các bước sau:

1. Nếu `teacher.finetune` bật, fine-tune teacher từ BEATs pretrain với FRCRN đóng băng, chọn epoch theo validation. Nếu không, nạp `teacher.checkpoint`. Sau đó cache logits của train và validation theo `sample_id`. Cache hợp lệ vì dataset cố định và không augment.
2. Với mỗi thế hệ, dựng student (seed riêng), huấn luyện với loss BAN, chọn epoch theo `macro_f1_snr` trên validation, rồi ghi `result` và `predictions`.
3. Nếu `baseline.enabled`, huấn luyện ensemble không chưng cất với cùng số seed.
4. **BANE**: lấy trung bình xác suất softmax của các thế hệ. Đây là đầu ra ensemble để so sánh với từng student.

Cách triển khai theo BAN: student khởi tạo ngẫu nhiên (`student_init: scratch`), mỗi thế hệ chưng cất từ **thế hệ trước** (thế hệ k+1 dùng student thế hệ k làm teacher, đóng băng). Logit teacher của mỗi thế hệ được tính lại trên tập train.

Ô dưới chạy toàn bộ pipeline. Với `RUN_TRAINING = False` đây là smoke test một epoch trên 64 mẫu.

In [ ]:
# (7) Chạy pipeline: cache logits teacher, huấn luyện các thế hệ, đánh giá, BANE
result = run_pipeline(
    config,
    output_dir=OUTPUT_DIR,
    device=str(device),
    smoke=False,
    run_id=RUN_ID,
)
print("run:", result.run.run_id)
print("kết quả:", result.run.result_file)

## (8) Bảng kết quả và biểu đồ theo SNR

Bảng có một dòng cho teacher, một dòng cho mỗi thế hệ student, dòng `bane` và, nếu bật, dòng `baseline_ens`. Các chỉ số `*_hi` là trung bình trên các mức SNR từ 15 dB trở lên, vùng nhãn đáng tin hơn.

Đọc kết quả theo thứ tự: so teacher với student để thấy chưng cất có vượt teacher không, rồi so BANE với student đơn để thấy lợi ích của ensemble.

In [ ]:
# (8) Bảng kết quả và biểu đồ top-1 và macro-F1 theo SNR
from IPython.display import Image, Markdown, display

from scripts.run_baselines import markdown_table

columns = ("model", "top1_accuracy", "macro_f1", "macro_f1_snr", "top1_accuracy_hi", "macro_f1_hi")
display(Markdown(markdown_table(result.table(), columns)))
for figure in sorted(result.run.figure_dir.glob("*.png")):
    display(Image(str(figure)))

## Lưu ý trước khi đọc kết quả

1. **Rò rỉ dữ liệu.** `BEATs_iter3_plus_AS2M.pt` đã pretrain trên AudioSet 2M, và teacher `best.pt` được huấn luyện trên mixture có nhiễu AudioSet. Nếu clip nhiễu của validation nằm trong dữ liệu pretrain, con số validation sẽ cao hơn thực tế. Notebook `finetune-beat.ipynb` đã có bước lọc `validation_clean` cho việc này; notebook này chưa áp dụng.
2. **Teacher quá tự tin trên train.** Teacher được huấn luyện trên chính tập train, nên soft target trên train gần one-hot và chưng cất gần như không thêm thông tin. Cần đo entropy của `p_t` trên train trước khi kết luận về λ và T.
3. **Chuỗi thế hệ chưa đúng tài liệu.** Như mô tả ở bước (7), các thế hệ đang chưng cất từ teacher cố định.
4. **Baseline đang tắt.** Để đánh giá BAN có ý nghĩa, cần bật `baseline` (ensemble không chưng cất, cùng số model) và so sánh với BANE.
5. **Chọn mô hình chỉ trên validation.** Tập `test` chưa được nạp trong notebook này. Chỉ mở test sau khi cấu hình đã cố định.